In [1]:
import torch
import torch.nn as nn

In [99]:
class ConvBlock(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()        
        self.conv = nn.Conv2d(in_channels=in_channels, out_channels=out_channels, kernel_size=3, padding='same', dilation=(1,2))
        self.bn = nn.BatchNorm2d(num_features=out_channels)
        self.activation = nn.ReLU()

    def forward(self, x):
        return self.activation(self.bn(self.conv(x)))

class ChromaCNN(nn.Module):
    def __init__(self, conv_layers, conv_in_channels, pool, fc_layers, fc_in_channels, dropout, num_classes):
        super().__init__()
        self.conv_layers = self.get_conv_layers(conv_layers, conv_in_channels, pool)
        self.fc_layers = self.get_fc_layers(fc_layers, fc_in_channels, dropout, num_classes)

    def get_conv_layers(self, conv_layers, in_channels, pool):
        layers = []
        for val in conv_layers:
            if val=="M": 
                layers.append(nn.MaxPool2d(kernel_size=eval(pool)))
            else: 
                layers.append(ConvBlock(in_channels=in_channels, out_channels=val))
                in_channels = val
        return nn.ModuleList(layers)

    def get_fc_layers(self, fc_layers, in_channels, dropout, num_classes):
        layers = []
        in_channels  = eval(in_channels)
        for val in fc_layers:
            if val=="D": layers.append(nn.Dropout(dropout))
            elif val=="relu": layers.append(nn.ReLU())
            elif val=="gelu": layers.append(nn.GELU())
            else:
                layers.append(nn.Linear(in_features=in_channels, out_features=val))
                in_channels = val
        layers.append(nn.Linear(in_features=in_channels, out_features=num_classes))
        return nn.ModuleList(layers)
    
    def forward(self, x):
        for layer in self.conv_layers: x = layer(x)
        b, c, f, t = x.shape
        x = x.permute(0,3,1,2)
        x = x.reshape(b, t, -1)
        for layer in self.fc_layers: x = layer(x)
        return x


In [100]:
from omegaconf import OmegaConf

config_path = "/Users/jcastle/workspace/pansori/opt/configs/models/ChromaCNN.yaml"
config = OmegaConf.load(config_path)
model_params = OmegaConf.to_container(config.model)
model = ChromaCNN(**model_params)

In [101]:
model

ChromaCNN(
  (conv_layers): ModuleList(
    (0): ConvBlock(
      (conv): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=same, dilation=(1, 2))
      (bn): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): ReLU()
    )
    (1): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (2): ConvBlock(
      (conv): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=same, dilation=(1, 2))
      (bn): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): ReLU()
    )
    (3): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (4): ConvBlock(
      (conv): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=same, dilation=(1, 2))
      (bn): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (activation): ReLU()
    )
    (5): MaxPool2d(kernel_size=(2, 1), stride=(2, 1

In [103]:
model(torch.Tensor(size=(1, 1, 25,625))).shape

torch.Size([1, 625, 4])

In [ ]:
@hydra.main(config_path=".", config_name="ChromaCNN")
def main(cfg: DictConfig):
    # 모델 구성 파라미터 추출
    model_config = cfg.model
    
    # ChromaCNN 모델 초기화
    model = ChromaCNN(
        conv_layers=model_config.conv_layers,
        conv_in_channels=model_config.conv_in_channels,
        kernel_size=model_config.kernel_size,
        padding=model_config.padding,
        dilation=model_config.dilation,
        pool=model_config.pool,
        fc_layers=model_config.fc_layers,
        fc_in_channels=model_config.fc_in_channels,
        dropout=model_config.dropout,
        num_classes=model_config.num_classes
    )
    
    print(model)
    
    # 여기에 학습 또는 추론 코드 추가
    
if __name__ == "__main__":
    main()